In [15]:
import warnings
warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", category=UserWarning)

import os, pickle
import re

import numpy as np
import matplotlib, matplotlib.pyplot as plt
from matplotlib import ticker, lines
import hist
import vector
import pandas as pd

from physics.simulation import msq, mcfm
from physics.analysis import zz2l2v, zz4l
from physics.hstar import eft
from physics.variables import eft_terms
from datasets import coefficient
from nsbi import taylr

import torch
from torch.utils.data import TensorDataset, DataLoader
import lightning as L

In [2]:
torch.set_float32_matmul_precision('high')

matplotlib.use("pgf")
matplotlib.rcParams.update({
    "pgf.texsystem": "lualatex",
    "text.usetex": True,
    "pgf.rcfonts": False,
    "pgf.preamble": "", 
})

In [3]:
run_dir_4l = 'run/presel/h4l'
run_dir_2l2v = 'run/presel/h2l2v'


c6_powers = [ eft_terms[i][0] for i in range(len(eft_terms)) ]
ct_powers = [ eft_terms[i][1] for i in range(len(eft_terms)) ]
cg_powers = [ eft_terms[i][2] for i in range(len(eft_terms)) ]

(_, events_gg_4l), scaler_X_taylr_4l, models_taylr_4l, scalers_y_taylr_4l = taylr.utils.load_results(run_dir_4l, eft_terms)
(_, events_gg_2l2v), scaler_X_taylr_2l2v, models_taylr_2l2v, scalers_y_taylr_2l2v = taylr.utils.load_results(run_dir_2l2v, eft_terms)

In [4]:
features_4l = ['l1_pt', 'l1_eta', 'l1_phi', 'l1_energy',
                'l2_pt', 'l2_eta', 'l2_phi', 'l2_energy',
                'l3_pt', 'l3_eta', 'l3_phi', 'l3_energy',
                'l4_pt', 'l4_eta', 'l4_phi', 'l4_energy']
features_2l2v = ["l1_pt", "l1_eta", "l1_phi", "l1_energy", "l2_pt", "l2_eta", "l2_phi", "l2_energy", "met", "met_phi"]

batch_size = 1024

In [5]:
# Determine shape of the 3D arrays
max_n = max(c6_powers) + 1
max_m = max(ct_powers) + 1
max_l = max(cg_powers) + 1

testing_data_4l = scaler_X_taylr_4l.transform(events_gg_4l.kinematics[features_4l].to_numpy())
testing_data_2l2v = scaler_X_taylr_2l2v.transform(events_gg_2l2v.kinematics[features_2l2v].to_numpy())

testing_data_4l = DataLoader(TensorDataset(torch.tensor(testing_data_4l,dtype=torch.float32)),batch_size=batch_size)
testing_data_2l2v = DataLoader(TensorDataset(torch.tensor(testing_data_2l2v,dtype=torch.float32)),batch_size=batch_size)


In [6]:
trainer = L.Trainer(accelerator='gpu', devices=1)

coeffs_4l = [[[None for _ in range(max_l)] for _ in range(max_m)] for _ in range(max_n)]
coeffs_2l2v = [[[None for _ in range(max_l)] for _ in range(max_m)] for _ in range(max_n)]

for ctup in eft_terms:
    i, j, k = ctup

    output_4l = torch.cat(trainer.predict(models_taylr_4l[i][j][k], testing_data_4l))
    coeff_4l = scalers_y_taylr_4l[i][j][k].inverse_transform(output_4l.numpy().reshape(-1, 1)).reshape(-1)
    coeffs_4l[i][j][k] = coeff_4l

    output_2l2v = torch.cat(trainer.predict(models_taylr_2l2v[i][j][k], testing_data_2l2v))
    coeff_2l2v = scalers_y_taylr_2l2v[i][j][k].inverse_transform(output_2l2v.numpy().reshape(-1, 1)).reshape(-1)
    coeffs_2l2v[i][j][k] = coeff_2l2v

# populate (0,0,0) with 1
coeffs_4l[0][0][0] = np.ones_like(coeffs_4l[1][0][0])
coeffs_2l2v[0][0][0] = np.ones_like(coeffs_2l2v[1][0][0])

# populate any other None values with 0
for i in range(max_n):
    for j in range(max_m):
        for k in range(max_l):
            if coeffs_4l[i][j][k] is None:
                coeffs_4l[i][j][k] = np.zeros_like(coeffs_4l[1][0][0])
            if coeffs_2l2v[i][j][k] is None:
                coeffs_2l2v[i][j][k] = np.zeros_like(coeffs_2l2v[1][0][0])

# reshape to (Nevents, eft, ct, cg)
coeffs_4l = np.array(coeffs_4l).transpose((3,0,1,2))
coeffs_2l2v = np.array(coeffs_2l2v).transpose((3,0,1,2))

Using default `ModelCheckpoint`. Consider installing `litmodels` package to enable `LitModelCheckpoint` for automatic upload to the Lightning model registry.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
The following callbacks returned in `LightningModule.configure_callbacks` will override existing callbacks passed to Trainer: ModelCheckpoint
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1,2,3,4,5,6,7]


Predicting: |                                                                                                 …

The following callbacks returned in `LightningModule.configure_callbacks` will override existing callbacks passed to Trainer: EarlyStopping, ModelCheckpoint
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1,2,3,4,5,6,7]


Predicting: |                                                                                                 …

The following callbacks returned in `LightningModule.configure_callbacks` will override existing callbacks passed to Trainer: EarlyStopping, ModelCheckpoint
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1,2,3,4,5,6,7]


...

Predicting: |                                                                                                 …

The following callbacks returned in `LightningModule.configure_callbacks` will override existing callbacks passed to Trainer: EarlyStopping, ModelCheckpoint
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1,2,3,4,5,6,7]


Predicting: |                                                                                                 …

The following callbacks returned in `LightningModule.configure_callbacks` will override existing callbacks passed to Trainer: EarlyStopping, ModelCheckpoint
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1,2,3,4,5,6,7]


Predicting: |                                                                                                 …

In [62]:
p_sm = events_gg_4l.weights / events_gg_4l.weights.sum()
coeffs = coeffs_4l
eft_mod = eft.Modifier(baseline=msq.Component.SBI, events=events_gg_4l, c6_points=[-20,-10,0,10,20])
xobs = events_gg_4l.calculate(zz4l.FourLeptonSystem()).kinematics['4l_mass']
xbins = np.concatenate([np.arange(180,250,10), np.arange(250,500,25), np.arange(500,1000,50), np.arange(1000,1300,100)])
xmin, xmax = np.min(xbins), np.max(xbins)
xwidths = np.diff(xbins)
xcenters = 0.5 * (xbins[:-1] + xbins[1:])
xlabel='$m_{ZZ}\\ \\mathrm{[GeV]}$'
xname = 'mzz'

# p_sm = events_gg_2l2v.weights / events_gg_2l2v.weights.sum()
# coeffs = coeffs_2l2v
# eft_mod = eft.Modifier(baseline=msq.Component.SBI, events=events_gg_2l2v, c6_points=[-20,-10,0,10,20])
# xobs = events_gg_2l2v.calculate(zz2l2v.ZZ2L2V()).kinematics['zz_mt']
# xbins = np.concatenate([np.arange(250,300,10), np.arange(300,400,25), np.arange(400,600,50), np.arange(600,900,150) , np.arange(900,1300,300)])
# xmin, xmax = np.min(xbins), np.max(xbins)
# xwidths = np.diff(xbins)
# xcenters = 0.5 * (xbins[:-1] + xbins[1:])
# xlabel='$m_{T}^{ZZ}\\ \\mathrm{[GeV]}$'
# xname = 'mtzz'

In [63]:
from physics.constants import cH_to_c6, ctH_to_ct, cHG_to_cg
cH_vals=np.array([50.0,-50.0])
ctH_vals=np.array([10.0,-10.0])
cHG_vals=np.array([0.1,-0.1])
c6_vals=cH_vals * cH_to_c6
ct_vals=ctH_vals * ctH_to_ct
cg_vals=cHG_vals * cHG_to_cg

In [64]:
eft_mod.coefficients[:,1:,1:,1:] = np.zeros_like(eft_mod.coefficients[:,1:,1:,1:])
eft_mod.coefficients[:,2:,2:,:] = np.zeros_like(eft_mod.coefficients[:,2:,2:,:])
eft_mod.coefficients[:,2:,:,2:] = np.zeros_like(eft_mod.coefficients[:,2:,:,2:])

w_eft, p_eft = eft_mod.modify(c6=c6_vals, ct=ct_vals, cg=cg_vals)

In [65]:
w_eft_up, p_eft_up = w_eft[:,0,0,0], p_eft[:,0,0,0]
w_eft_dn, p_eft_dn = w_eft[:,1,1,1], p_eft[:,1,1,1]

In [66]:
c6_powers = np.stack([np.power(c6_vals,i) for i in range(4+1)], axis=0)   # (5, Nx)
ct_powers = np.stack([np.power(ct_vals,j) for j in range(2+1)], axis=0)   # (3, Ny)
cg_powers = np.stack([np.power(cg_vals,k) for k in range(2+1)], axis=0)   # (3, Nz)

msq_bsm_over_sm = np.einsum('nijk,ix,jy,kz->nxyz', coeffs, c6_powers, ct_powers, cg_powers)
reweight_up = msq_bsm_over_sm[:,0,0,0]
reweight_dn = msq_bsm_over_sm[:,1,1,1]

In [69]:
lw = 1.25

h_true_up = hist.Hist(hist.axis.Variable(xbins), storage=hist.storage.Weight())
h_true_dn = hist.Hist(hist.axis.Variable(xbins), storage=hist.storage.Weight())
h_true_up.fill(xobs, weight=p_eft_up)
h_true_dn.fill(xobs, weight=p_eft_dn)

h_sm = hist.Hist(hist.axis.Variable(xbins), storage=hist.storage.Weight())
h_sm.fill(xobs, weight=p_sm)

p_rwt_up = p_sm * reweight_up / np.sum(p_sm * reweight_up)
h_reweight_up = hist.Hist(hist.axis.Variable(xbins), storage=hist.storage.Weight())
h_reweight_up.fill(xobs, weight=p_rwt_up)

p_rwt_dn = p_sm * reweight_dn / np.sum(p_sm * reweight_dn)
h_reweight_dn = hist.Hist(hist.axis.Variable(xbins), storage=hist.storage.Weight())
h_reweight_dn.fill(xobs, weight=p_rwt_dn)

fig, (ax1, ax2, ax3) = plt.subplots(3,1,gridspec_kw={'height_ratios': [3, 1, 1]},figsize=(5,6), layout='constrained', sharex=True)
plt.subplots_adjust(wspace=0, hspace=0)

# Plot the stairs without labels
ax1.stairs(h_sm.values()/xwidths, h_sm.axes[0].edges, color='black', linestyle='--', linewidth=lw)
ax1.stairs(h_reweight_up.values()/xwidths, xbins, color='cornflowerblue',  linestyle='-', linewidth=lw)
ax1.stairs(h_true_up.values()/xwidths, xbins, color='blue', linestyle=':', linewidth=lw)
ax1.stairs(h_reweight_dn.values()/xwidths, h_reweight_up.axes[0].edges, color='tomato',  linestyle='-', linewidth=lw)
ax1.stairs(h_true_dn.values()/xwidths, xbins, color='red', linestyle=':', linewidth=lw)

legend_lines = [
    lines.Line2D([], [], color='black', linestyle='--', linewidth=lw, label='$\\mathrm{SM}$'),
    lines.Line2D([], [], color='grey', linestyle='-', linewidth=lw, label=f'$\\mathrm{{BSM}}\\ \\mathrm{{(MC)}}$'),
    lines.Line2D([], [], color='black', linestyle=':', linewidth=lw, label='$\\mathrm{{BSM}}\\ \\mathrm{{(NSBI)}}$'),
    # lines.Line2D([], [], color='cornflowerblue', linestyle='-', linewidth=lw, label=f'$C_{{H,tH,HG}} = {cH_vals[0]}, {ctH_vals[0]}, {cHG_vals[0]}\ \mathrm{{(MC)}}$'),
    # lines.Line2D([], [], color='tomato', linestyle='-', linewidth=lw, label=f'$C_{{H,tH,HG}} = {cH_vals[1]}, {ctH_vals[1]}, {cHG_vals[1]}\ \mathrm{{(MC)}}$'),
]

ax1.legend(handles=legend_lines, frameon=False, fontsize=12, ncols=1, loc='center right', bbox_to_anchor=(1.0,0.6))

plt.text(0.96 ,0.96, '$C_H = +50, C_{tH} = +10, C_{HG} = +0.01$', transform=ax1.transAxes, ha='right', va='top', fontsize=12, color='blue')
plt.text(0.96 ,0.86, '$C_H = -50, C_{tH} = -10, C_{HG} = -0.01$', transform=ax1.transAxes, ha='right', va='top', fontsize=12, color='red')

ax1.set_yscale('log')
ax1.set_ylabel('$\\mathrm{Density\\ of\\ events\\ [1/GeV]}$', loc='top', fontsize=15)
ax1.set_ylim(1e-6, 1e-2)
ax1.yaxis.set_ticks([1e-6, 1e-5, 1e-4, 1e-3, 1e-2, 1e-1])
ax1.yaxis.set_ticklabels([ '', '$10^{-5}$', '$10^{-4}$', '$10^{-3}$', '$10^{-2}$', '$10^{-1}$'])

# ax1.text(0.04 ,0.12, '$gg \\rightarrow (h^{\\ast} \\rightarrow) ZZ \\rightarrow 2\\ell 2 \\nu$', transform=ax1.transAxes, ha='left', va='bottom', fontsize=12)
ax1.text(0.04 ,0.12, '$gg \\rightarrow (h^{\\ast} \\rightarrow) ZZ \\rightarrow 4\\ell$', transform=ax1.transAxes, ha='left', va='bottom', fontsize=12)
ax1.text(0.04 ,0.04, '$\\sqrt{s} = 14\\,\\mathrm{TeV}$', transform=ax1.transAxes, ha='left', va='bottom', fontsize=12)

y = h_sm.values()
yerr = np.sqrt(h_sm.variances())
# ax2.fill_between(h_true_up.axes[0].centers, (y-yerr)/y, (y+yerr)/y, linewidth=0, color='black', alpha=0.1, step='mid')
ax2.stairs(h_sm.values()/h_sm.values(), h_sm.axes[0].edges, color='black', linewidth=lw, linestyle='--')

ax2.stairs(h_reweight_up.values()/h_sm.values(), xbins, color='cornflowerblue', linestyle='-', linewidth=lw)
ax2.stairs(h_true_up.values()/h_sm.values(), xbins, color='blue', linestyle=':', linewidth=lw)
ax2.stairs(h_reweight_dn.values()/h_sm.values(), xbins, color='salmon', linestyle='-', linewidth=lw)
ax2.stairs(h_true_dn.values()/h_sm.values(), xbins, color='red', linestyle=':', linewidth=lw)

ax1.tick_params(labelsize=12)
ax2.tick_params(labelsize=12)
ax3.tick_params(labelsize=12)

ax2.tick_params(top=True, labeltop=False)
ax2.set_xlabel('')
ax2.set_xticklabels([])
ax2.set_ylabel('$\\mathrm{BSM} / \\mathrm{SM}$', fontsize=15)
if xname=='mzz':
    ax2.set_ylim(0.0,10.0)
    ax2.yaxis.set_ticks([1.0,5.0,10.0])
elif xname=='mtzz':
    ax2.set_ylim(0.5,2.5)
    ax2.yaxis.set_ticks([0.5,1.0,2.0,2.5])

ax3.stairs(h_true_up.values()/h_true_up.values(), h_true_up.axes[0].edges, color='black', linewidth=lw, linestyle=':')
ax3.errorbar(xcenters, h_reweight_up.values()/h_true_up.values(), xerr=xwidths/2, yerr=np.sqrt(h_reweight_up.variances()/h_true_dn.values()), color='cornflowerblue', linewidth=lw, linestyle='none')
ax3.errorbar(xcenters, h_reweight_dn.values()/h_true_dn.values(), xerr=xwidths/2, yerr=np.sqrt(h_reweight_dn.variances()/h_true_dn.values()), color='salmon', linewidth=lw, linestyle='none')

ax3.set_xlabel(xlabel, loc='right', fontsize=15)
ax3.set_xlim(xmin, xmax)
ax3.set_xscale('linear')
if xname=='mzz':
    ax3.xaxis.set_ticks([180,250,500,1000,1200])
    ax3.xaxis.set_minor_locator(ticker.FixedLocator([190,200,210,220,230,240,275,300,325,350,375,400,425,450,475,550,600,650,700,750,800,850,900,950,1100,1200]))
elif xname == 'mtzz':
    ax3.xaxis.set_ticks([250, 400, 600, 900,1200])
    ax3.xaxis.set_minor_locator(ticker.FixedLocator([260,270,280,290,300,325,350,375,450,500,550,750]))
# ax3.xaxis.set_ticks([180, 250, 500, 750, 1000])

ax3.set_ylabel('$\\mathrm{NSBI} / \\mathrm{MC}$', loc='center', fontsize=15)
ax3.set_ylim(0.96,1.04)
ax3.yaxis.set_ticks([0.98,1.0,1.02])
ax3.tick_params(top=True, labeltop=False)

plt.tight_layout()
plt.subplots_adjust(hspace=0)

plt.savefig(f'taylr_reweight_{xname}.pdf', bbox_inches='tight')
plt.close()